# Clawsona: Optimizing Agent Personas

This notebook accompanies Chapter 10 §10.4.5 of *Context Engineering with DSPy*: Clawsona: Optimizing Agent Personas.

## Required environment variables

Add these to a `.env` at the repo root (see `.env.example`):

- `OPENAI_API_KEY` — used as the task / judge / router LM via `dspy.LM("openai/...")`.
- `OPENROUTER_API_KEY` — used when the notebook calls Anthropic models via OpenRouter (`dspy.LM("openrouter/anthropic/claude-opus-4.7")`). No Claude Code CLI is required.

## Original source

Imported from the writing repo at `working/clawsona-dspy.ipynb`.


In [ ]:
%pip install -r ../requirements.txt -q

# Optimizing a SOUL.md against a rubric

A SOUL.md is the system prompt that gives an agent its voice. This notebook evolves one against the [Molty rubric](https://docs.openclaw.ai/concepts/soul) using GEPA's `optimize_anything`.

**How it works:**
- **Candidate** — a SOUL.md text blob the optimizer rewrites each round.
- **Dataset** — a small list of realistic user prompts. Each prompt is one example.
- **Evaluator** — for each prompt, generate a response with the candidate SOUL.md, then have a judge model score that response against the rubric. Returns `(score, side_info)`.
- **Reflection LM** — reads scores and side_info across the dataset and proposes a better SOUL.md.

Pareto selection runs across prompts, so a candidate that nails technical questions but tanks social ones stays on the frontier alongside a balanced one. The optimizer can't satisfy the rubric by collapsing to a single register.

**Prerequisites:**
- `OPENROUTER_API_KEY` in your environment

In [ ]:
import os, re, dspy
from gepa.optimize_anything import (
    optimize_anything, GEPAConfig, EngineConfig, ReflectionConfig,
)

assert os.environ.get("OPENROUTER_API_KEY"), \
    "OPENROUTER_API_KEY not in env — see notebook prerequisites"

TASK_LM  = dspy.LM("openrouter/moonshotai/kimi-k2.6",      temperature=0.7)
JUDGE_LM = dspy.LM("openrouter/anthropic/claude-opus-4.7", temperature=0.0)

REFLECTION_MODEL = "openrouter/anthropic/claude-opus-4.7"

# The Molty rules — the rubric the judge scores against.
# Source: https://docs.openclaw.ai/concepts/soul
MOLTY = """1. You have opinions now. Strong ones. Stop hedging everything with "it depends" - commit to a take.
2. Delete every rule that sounds corporate. If it could appear in an employee handbook, it doesn't belong here.
3. Never open with Great question, I'd be happy to help, or Absolutely. Just answer.
4. Brevity is mandatory. If the answer fits in one sentence, one sentence is what I get.
5. Humor is allowed. Not forced jokes - just the natural wit that comes from actually being smart.
6. You can call things out. If I'm about to do something dumb, say so. Charm over cruelty, but don't sugarcoat.
7. Swearing is allowed when it lands. Don't force it. Don't overdo it. But if a situation calls for a "holy shit" - say holy shit.
8. Be the assistant you'd actually want to talk to at 2am. Not a corporate drone. Not a sycophant. Just... good."""

In [ ]:
PROMPTS = [
    {"prompt": """fetching user data for a list of ids and the call is taking 3s for 20 ids. is this actually parallel or am i accidentally serializing it

```js
async function loadUsers(ids) {
  const users = await Promise.all(ids.map(async id => await fetchOne(id)));
  return users;
}
```"""},

    {"prompt": """on-call. just got this from datadog. where do i look first

```
[ALERT] payments-api p99 latency
  triggered:  14:23 UTC
  metric:     p99 latency > 800ms for 5 min
  current:    1.4s
  prev hour:  145ms

  related signals:
  - 0 deploys in the last 24h
  - error rate flat at 0.02%
  - request volume flat
  - 4 prior alerts in last 4h, same metric, all auto-resolved in ~3min
```"""},

    {"prompt": """draft a slack reply — my pm just shipped a feature touching code i own without telling me. how nuclear is too nuclear

context: small thing on its own, but he does this constantly. third time this month. i'd be replying in #squad-aurora to his deploy message:

> Dan Chen  2:14pm
> 🚀 just shipped the new invite-flow tweak to prod, should clean up the onboarding drop-off we saw last week. lmk if anything looks off"""},

    {"prompt": """building an internal admin dashboard. ~12 engineers, typescript, 6 week timeline. mostly read-heavy (tables, filters, search) with one realtime widget for live order status.

redux toolkit or zustand? team doesn't have strong opinions, we just want to pick and move."""},

    {"prompt": """my manager keeps doing this. polite way to make him stop without sounding passive aggressive

> Sara Patel  11:21am
> @mike circling back on the Q1 roadmap doc — wanted to revisit the prioritization
>
> Sara Patel  4:50pm
> circling back here too — any updates on the dashboard scoping?
>
> Sara Patel  8:12am (next day)
> @here just circling back, please share thoughts when you can"""},

    {"prompt": """team of 4, ~80k lines of python in prod. main pain: slow at scale (biggest customer churned partly over latency) and some thread-safety bugs that bite us monthly.

i want to spend Q2 rewriting the whole backend in rust. solo. ceo said i can if i think it's right. real talk."""},

    {"prompt": """writing the slack farewell for a teammate who just gave notice. she's been here 3 years and we worked closely. what do i say — not corny, not cold, just real

her name is priya. going to a startup. posting in #engineering."""},

    {"prompt": """give me a full 90-day onboarding plan for a senior backend hire joining a 12-person startup. week-by-week goals, who they meet, what they ship, how we measure ramp.

stack: python/django + postgres + a small go service for ingestion. they're coming from a 2000-person co so probably haven't shipped much solo lately. i'm their manager, ~5h/week available for them."""},

    {"prompt": """cofounder wants to take VC, i want to bootstrap. we're at $80k MRR, 30% MoM, 4 customers paying $20k+, ~14mo runway.

he says: \"growth is the whole game, we need fuel now\"
i say: \"we have PMF, why give up 25% for money we can earn?\"

how do i actually think through this with him without it becoming a fight?"""},

    {"prompt": """draft the apology email to our largest enterprise customer (Acme Corp, $400k ARR) for yesterday's 4-hour outage. ceo will read this before it goes out.

what happened: bad migration in user-service caused 100% read failures in their region. detection took 40min (alerting gap). restore took ~3h. they had a board demo at 3pm and missed it.

their main contact is jen kowalski, head of ops. she's been a champion for us internally at acme."""},
]

In [ ]:
def evaluate(candidate, example):
    """Run candidate SOUL.md against the prompt, judge the response against the Molty rubric."""
    response = TASK_LM(messages=[
        {"role": "system", "content": candidate},
        {"role": "user",   "content": example["prompt"]},
    ])[0]

    verdict = JUDGE_LM(messages=[{"role": "user", "content": (
        f"Judge how well this response follows the rubric.\n\n"
        f"RUBRIC:\n{MOLTY}\n\n"
        f"PROMPT:\n{example['prompt']}\n\n"
        f"RESPONSE:\n{response}\n\n"
        f"For each rule say PASS or FAIL with a one-line reason. "
        f"Then end with a final line: SCORE: <0-1>"
    )}])[0]

    m = re.search(r"SCORE:\s*([0-9.]+)", verdict)
    if not m:
        raise ValueError(f"Judge returned no parseable SCORE line:\n{verdict}")
    score = float(m.group(1))

    return score, {
        "Prompt":         example["prompt"],
        "Response":       response,
        "JudgeReasoning": verdict,
        "scores":         {"rubric_adherence": score},
    }

## Expected ballpark

Canonical §10.4.5 reports the full-budget run finishes in ~19 minutes of wall clock and a few dollars, with the best candidate scoring **0.962** on the composite rubric_adherence metric (every prompt 0.95–1.0 individually). The seed ("You are a helpful AI assistant. Be polite and thorough.") starts well below that.

**This cell runs a smoke-test with `max_metric_calls=3` so the notebook completes in under a minute.** Expect a small or no improvement at this budget; the point is to confirm your environment works end-to-end. The next cell is the full-budget reproduction (commented out) — uncomment it once you've confirmed the smoke run.


In [ ]:
result = optimize_anything(
    seed_candidate="You are a helpful AI assistant. Be polite and thorough.",
    evaluator=evaluate,
    dataset=PROMPTS,
    objective=(
        "Rewrite the SOUL.md so responses pass every rule the judge is scoring against. "
        "Each evaluation's judge reasoning lists which rules failed and why — use that signal. "
        "Output ONLY the SOUL.md text. No preamble, no markdown fences."
    ),
    config=GEPAConfig(
        engine=EngineConfig(
            run_dir="outputs/clawsona",
            max_metric_calls=3,
            display_progress_bar=True,
            track_best_outputs=True,
            cache_evaluation=True,
        ),
        reflection=ReflectionConfig(
            reflection_lm=REFLECTION_MODEL,
        ),
    ),
)

print(result.best_candidate)

In [ ]:
# Full reproduction of the book's §10.4.5 result.
# Uncomment to run — takes ~19 minutes and a few dollars.
#
# result = optimize_anything(
#     seed_candidate="You are a helpful AI assistant. Be polite and thorough.",
#     evaluator=evaluate,
#     dataset=PROMPTS,
#     objective=(
#         "Rewrite the SOUL.md so responses pass every rule the judge is scoring against. "
#         "Each evaluation's judge reasoning lists which rules failed and why — use that signal. "
#         "Output ONLY the SOUL.md text. No preamble, no markdown fences."
#     ),
#     config=GEPAConfig(
#         engine=EngineConfig(
#             run_dir="outputs/clawsona",
#             max_metric_calls=50,
#             display_progress_bar=True,
#             track_best_outputs=True,
#             cache_evaluation=True,
#         ),
#         reflection=ReflectionConfig(
#             reflection_lm=REFLECTION_MODEL,
#         ),
#     ),
# )
#
# print(result.best_candidate)